## Streamlit Dashboard

Writes `app.py` to disk, multi-tab dashboard with:
- Well selector + KPI cards (qi, b-factor, EUR, RMSE comparison)
- Production forecast chart: actual vs. classical Arps vs. XGBoost, with anomalies marked
- Water cut & GOR trend charts
- Log-log and rate-cumulative diagnostic plots
- Anomaly table with likely-cause classification
- Field-wide model comparison and feature importance

In [ ]:
%%writefile app.py
import duckdb
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from plotly.subplots import make_subplots

st.set_page_config(page_title="Smart DCA Dashboard", page_icon="🏭", layout="wide")

# ---------------------------------------------------------------- palette
# Modified for a light/white background with high-contrast text and warm oil accents.
OIL_BLACK   = "#1a1310"
RUST        = "#c2410c"
AMBER       = "#d97706"  # Darkened slightly for better readability on white
COPPER      = "#b45309"
OLIVE       = "#60643a"  # Darkened for text contrast
BG_WHITE    = "#ffffff"
TEXT_DARK   = "#1f2937"  # Dark gray for crisp body text readability
BORDER_GRAY = "#e5e7eb"
DEEP_RUST   = "#7c2d12"

st.markdown(f"""
<style>
    :root {{
        --oil-black: {OIL_BLACK};
        --rust: {RUST};
        --amber: {AMBER};
        --copper: {COPPER};
        --text-dark: {TEXT_DARK};
        --bg-white: {BG_WHITE};
        --border-gray: {BORDER_GRAY};
    }}

    .stApp {{
        background: {BG_WHITE};
        color: var(--text-dark) !important;
    }}
    html, body, [class*="css"] {{ color: var(--text-dark) !important; }}

    /* Clear header backgrounds */
    header[data-testid="stHeader"] {{
        background-color: transparent !important;
        background: transparent !important;
    }}
    div[data-testid="stDecoration"] {{ display: none !important; }}
    header[data-testid="stHeader"] * {{ color: var(--text-dark) !important; fill: var(--text-dark) !important; }}

    h1, h2, h3, h4 {{ color: var(--copper) !important; font-weight: 800 !important; }}
    p, span, label, div {{ color: var(--text-dark); }}

    /* Light sidebar styling */
    section[data-testid="stSidebar"] {{
        background: #f9fafb;
        border-right: 1px solid var(--border-gray);
    }}
    section[data-testid="stSidebar"] * {{ color: var(--text-dark) !important; }}

    .stSelectbox label, .stCheckbox label, .stCaption, [data-testid="stCaptionContainer"] {{
        color: var(--text-dark) !important;
        opacity: 0.9;
    }}

    /* Selectbox light theme adjustments */
    div[data-baseweb="select"] {{ background-color: #ffffff !important; }}
    div[data-baseweb="select"] > div {{
        background-color: #ffffff !important;
        border-color: var(--border-gray) !important;
        color: var(--text-dark) !important;
    }}
    div[data-baseweb="select"] * {{ color: var(--text-dark) !important; background-color: transparent !important; }}
    div[data-baseweb="select"] svg {{ fill: var(--text-dark) !important; }}
    ul[data-testid="stSelectboxVirtualDropdown"] {{ background-color: #ffffff !important; border: 1px solid var(--border-gray); }}
    ul[data-testid="stSelectboxVirtualDropdown"] li {{ background-color: #ffffff !important; color: var(--text-dark) !important; }}
    ul[data-testid="stSelectboxVirtualDropdown"] li:hover {{ background-color: #f3f4f6 !important; }}

    /* Tabs styling */
    button[data-baseweb="tab"] {{
        color: var(--text-dark) !important;
        font-weight: 600;
    }}
    button[data-baseweb="tab"][aria-selected="true"] {{
        color: var(--amber) !important;
        border-bottom: 3px solid var(--rust) !important;
    }}
    div[data-baseweb="tab-highlight"] {{ background-color: var(--rust) !important; }}

    [data-testid="stAppViewContainer"] > .main .block-container {{
        padding-top: 1.5rem !important;
    }}
    button[data-testid="stSidebarCollapseButton"], button[data-testid="baseButton-headerNoPadding"] {{
        color: var(--text-dark) !important;
    }}
    button[data-testid="stSidebarCollapseButton"] svg {{ fill: var(--text-dark) !important; }}

    /* Metric cards: Clean, light style with crisp borders */
    .metric-card {{
        background: #ffffff;
        border: 1px solid var(--border-gray);
        border-radius: 14px;
        padding: 18px 20px;
        text-align: center;
        box-shadow: 0 4px 12px rgba(0,0,0,0.05);
    }}
    .metric-card h3 {{ color: #6b7280 !important; font-size: 12.5px; font-weight: 600;
                       margin: 0 0 6px 0; text-transform: uppercase; letter-spacing: .6px;}}
    .metric-card p {{ color: var(--oil-black) !important; font-size: 27px; font-weight: 800; margin: 0; }}
    .badge-good {{ background:#dcfce7; color:#15803d; padding:3px 10px; border-radius:20px; font-size:12px; font-weight: 600;}}
    .badge-bad  {{ background:#fee2e2; color:#b91c1c; padding:3px 10px; border-radius:20px; font-size:12px; font-weight: 600;}}

    [data-testid="stDataFrame"] {{ border: 1px solid var(--border-gray); border-radius: 10px; }}

    hr {{ border-color: var(--border-gray) !important; }}

    /* Animated banner shifted to a crisp light layout */
    .rig-banner {{
        position: relative;
        height: 130px;
        margin-bottom: 6px;
        overflow: hidden;
        border-radius: 14px;
        background: linear-gradient(180deg, #f9fafb 0%, #f3f4f6 100%);
        border: 1px solid var(--border-gray);
    }}
    .rig-banner svg {{ position: absolute; bottom: 0; left: 0; }}
    .pumpjack-arm {{
        transform-origin: 140px 55px;
        animation: pump 2.4s ease-in-out infinite;
    }}
    @keyframes pump {{
        0%   {{ transform: rotate(-14deg); }}
        50%  {{ transform: rotate(14deg); }}
        100% {{ transform: rotate(-14deg); }}
    }}
    .pump-head {{
        transform-origin: 140px 55px;
        animation: pumphead 2.4s ease-in-out infinite;
    }}
    @keyframes pumphead {{
        0%   {{ transform: translateY(-6px); }}
        50%  {{ transform: translateY(6px); }}
        100% {{ transform: translateY(-6px); }}
    }}
    .oil-drop {{
        position: absolute;
        bottom: 100%;
        font-size: 20px;
        opacity: 0.85;
        animation: drop linear infinite;
    }}
    @keyframes drop {{
        0%   {{ transform: translateY(-10px); opacity: 0; }}
        15%  {{ opacity: 0.9; }}
        100% {{ transform: translateY(150px); opacity: 0; }}
    }}
</style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------------- animated banner
st.markdown(f"""
<div class="rig-banner">
  <svg width="100%" height="130" viewBox="0 0 900 130" preserveAspectRatio="xMidYMax slice">
    <rect x="0" y="118" width="900" height="12" fill="{RUST}" opacity="0.2"/>
    <g opacity="0.4" stroke="{COPPER}" stroke-width="3" fill="none">
      <path d="M700 118 L710 55 L720 118" />
      <path d="M703 85 L717 85" />
      <path d="M705 100 L715 100" />
      <path d="M760 118 L768 65 L776 118" />
      <path d="M762 90 L774 90" />
    </g>
    <g transform="translate(60,0)">
      <rect x="20" y="108" width="16" height="10" fill="{RUST}"/>
      <line x1="28" y1="108" x2="28" y2="55" stroke="{COPPER}" stroke-width="5"/>
      <line x1="10" y1="112" x2="46" y2="60" stroke="{COPPER}" stroke-width="4"/>
      <g class="pumpjack-arm">
        <line x1="0" y1="55" x2="80" y2="55" stroke="{AMBER}" stroke-width="6" stroke-linecap="round"/>
        <circle cx="0" cy="55" r="6" fill="{RUST}"/>
      </g>
      <g class="pump-head">
        <line x1="0" y1="55" x2="0" y2="100" stroke="{AMBER}" stroke-width="4"/>
      </g>
      <circle cx="28" cy="55" r="7" fill="{DEEP_RUST}"/>
    </g>
    <g transform="translate(830,88)" opacity="0.9">
      <rect x="0" y="0" width="28" height="30" rx="4" fill="{RUST}"/>
      <ellipse cx="14" cy="0" rx="14" ry="4" fill="{AMBER}"/>
      <text x="6" y="20" font-size="10" fill="#ffffff" font-weight="bold">OIL</text>
    </g>
  </svg>
  <span class="oil-drop" style="left:145px; animation-duration:2.6s;">🛢️</span>
  <span class="oil-drop" style="left:200px; animation-duration:3.4s; animation-delay:.8s; font-size:14px;">●</span>
  <span class="oil-drop" style="left:170px; animation-duration:3s; animation-delay:1.6s; font-size:12px;">●</span>
</div>
""", unsafe_allow_html=True)

@st.cache_resource
def get_con():
    return duckdb.connect("dca_warehouse.duckdb", read_only=True)

con = get_con()

@st.cache_data
def load_tables():
    gold = con.execute("SELECT * FROM gold_features ORDER BY well_id, date").df()
    arps_params = con.execute("SELECT * FROM arps_params").df()
    arps_forecast = con.execute("SELECT * FROM arps_forecast").df()
    ml_pred = con.execute("SELECT * FROM ml_predictions").df()
    comparison = con.execute("SELECT * FROM model_comparison").df()
    anomalies = con.execute("SELECT * FROM anomalies").df()
    importance = con.execute("SELECT * FROM feature_importance").df()
    meta = con.execute("SELECT * FROM well_metadata").df()
    return gold, arps_params, arps_forecast, ml_pred, comparison, anomalies, importance, meta

gold, arps_params, arps_forecast, ml_pred, comparison, anomalies, importance, meta = load_tables()

# ---------------------------------------------------------------- sidebar
st.sidebar.markdown("## 🛢️ Smart DCA Dashboard")
st.sidebar.caption("Decline Curve Analysis + Data Engineering + AI")
wells = sorted(gold["well_id"].unique())
well = st.sidebar.selectbox("Select Well", wells)
st.sidebar.markdown("---")
show_ml = st.sidebar.checkbox("Show ML (XGBoost) forecast", value=True)
show_arps = st.sidebar.checkbox("Show classical Arps forecast", value=True)
show_anomalies = st.sidebar.checkbox("Highlight anomalies", value=True)
st.sidebar.markdown("---")
st.sidebar.markdown("**Pipeline layers**")
st.sidebar.markdown("`bronze` → raw SCADA export\n\n`silver` → cleaned & gap-filled\n\n`gold` → feature-engineered")

well_meta = meta[meta["well_id"] == well].iloc[0]
well_gold = gold[gold["well_id"] == well].sort_values("date")
well_arps_params = arps_params[arps_params["well_id"] == well]
well_arps_fc = arps_forecast[arps_forecast["well_id"] == well]
well_ml = ml_pred[ml_pred["well_id"] == well]
well_comp = comparison[comparison["well_id"] == well]
well_anom = anomalies[anomalies["well_id"] == well]

# ---------------------------------------------------------------- header
st.markdown(f"# 🛢️ {well}")
st.caption(f"Spud date {well_meta['spud_date']}  |  Choke {int(well_meta['choke_size_64th'])}/64\"  |  "
           f"Lat {well_meta['latitude']:.3f}, Lon {well_meta['longitude']:.3f}")

if len(well_arps_params):
    p = well_arps_params.iloc[0]
    c1, c2, c3, c4, c5 = st.columns(5)
    with c1:
        st.markdown(f'<div class="metric-card"><h3>Initial Rate (qi)</h3><p>{p["qi_fit"]:,.0f} bbl/d</p></div>', unsafe_allow_html=True)
    with c2:
        st.markdown(f'<div class="metric-card"><h3>Decline (b-factor)</h3><p>{p["b_fit"]:.2f}</p></div>', unsafe_allow_html=True)
    with c3:
        st.markdown(f'<div class="metric-card"><h3>EUR (Arps)</h3><p>{p["eur_bbl"]/1000:,.0f}k bbl</p></div>', unsafe_allow_html=True)
    with c4:
        arps_rmse = p["holdout_rmse"]
        st.markdown(f'<div class="metric-card"><h3>Arps 90d RMSE</h3><p>{arps_rmse:.1f}</p></div>', unsafe_allow_html=True)
    with c5:
        if len(well_comp):
            ml_rmse = well_comp.iloc[0]["ml_holdout_rmse"]
            better = "badge-good" if ml_rmse < arps_rmse else "badge-bad"
            label = "ML wins" if ml_rmse < arps_rmse else "Arps wins"
            st.markdown(f'<div class="metric-card"><h3>XGBoost 90d RMSE</h3><p>{ml_rmse:.1f}</p>'
                        f'<span class="{better}">{label}</span></div>', unsafe_allow_html=True)

st.markdown("###")

tab1, tab2, tab3, tab4 = st.tabs(["📈 Production Forecast", "📉 Diagnostic (log-log)", "🚨 Anomalies", "🧠 Model Insights"])

# Plot templates flipped to a clear white/light configuration with light-gray grid lines
PLOT_TEMPLATE_LAYOUT = dict(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color=TEXT_DARK),
    xaxis=dict(gridcolor=BORDER_GRAY, zerolinecolor=BORDER_GRAY),
    yaxis=dict(gridcolor=BORDER_GRAY, zerolinecolor=BORDER_GRAY),
)

# ---------------------------------------------------------------- tab 1: forecast
with tab1:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=well_gold["date"], y=well_gold["oil_rate_bbl"],
        mode="lines", name="Actual (cleaned)", line=dict(color=AMBER, width=1.8)
    ))
    if show_arps and len(well_arps_fc):
        fc_dates = pd.to_datetime(well_gold["date"].iloc[0]) + pd.to_timedelta(well_arps_fc["days_on_prod"], unit="D")
        fig.add_trace(go.Scatter(
            x=fc_dates, y=well_arps_fc["arps_rate_bbl"],
            mode="lines", name="Classical Arps fit/forecast", line=dict(color=OLIVE, width=2, dash="dash")
        ))
    if show_ml and len(well_ml):
        fig.add_trace(go.Scatter(
            x=well_ml["date"], y=well_ml["ml_pred"],
            mode="markers", name="XGBoost forecast (holdout)", marker=dict(color="#b45309", size=6, symbol="diamond")
        ))
    if show_anomalies and len(well_anom):
        fig.add_trace(go.Scatter(
            x=well_anom["date"], y=well_anom["oil_rate_bbl"],
            mode="markers", name="Anomaly flagged", marker=dict(color=RUST, size=10, symbol="x")
        ))
    fig.update_layout(
        template=None, height=480, hovermode="x unified",
        margin=dict(l=10, r=10, t=30, b=10),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, font=dict(color=TEXT_DARK)),
        xaxis_title="Date", yaxis_title="Oil Rate (bbl/d)",
        **PLOT_TEMPLATE_LAYOUT,
    )
    st.plotly_chart(fig, use_container_width=True)

    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**Water cut trend**")
        fig_wc = go.Figure(go.Scatter(x=well_gold["date"], y=well_gold["water_cut"]*100,
                                       line=dict(color=RUST), fill="tozeroy", fillcolor="rgba(194,65,12,0.12)"))
        fig_wc.update_layout(height=260, margin=dict(l=10,r=10,t=10,b=10),
                              yaxis_title="Water cut (%)", **PLOT_TEMPLATE_LAYOUT)
        st.plotly_chart(fig_wc, use_container_width=True)
    with c2:
        st.markdown("**GOR trend**")
        fig_gor = go.Figure(go.Scatter(x=well_gold["date"], y=well_gold["gor_scf_bbl"],
                                        line=dict(color=COPPER), fill="tozeroy", fillcolor="rgba(180,83,9,0.12)"))
        fig_gor.update_layout(height=260, margin=dict(l=10,r=10,t=10,b=10),
                               yaxis_title="GOR (scf/bbl)", **PLOT_TEMPLATE_LAYOUT)
        st.plotly_chart(fig_gor, use_container_width=True)

# ---------------------------------------------------------------- tab 2: diagnostic log-log
with tab2:
    st.markdown("Classical rate-cumulative and log-log diagnostic plots used to sanity-check the decline regime.")
    d = well_gold[well_gold["oil_rate_bbl"] > 0]
    fig_diag = make_subplots(rows=1, cols=2, subplot_titles=("Rate vs Time (log-log)", "Rate vs Cumulative"))
    fig_diag.add_trace(go.Scatter(x=d["days_on_prod"]+1, y=d["oil_rate_bbl"], mode="markers",
                                   marker=dict(color=AMBER, size=4)), row=1, col=1)
    fig_diag.update_xaxes(type="log", title_text="Days on production", row=1, col=1)
    fig_diag.update_yaxes(type="log", title_text="Oil rate (bbl/d)", row=1, col=1)

    fig_diag.add_trace(go.Scatter(x=d["cum_oil_bbl"], y=d["oil_rate_bbl"], mode="markers",
                                   marker=dict(color=RUST, size=4)), row=1, col=2)
    fig_diag.update_xaxes(title_text="Cumulative oil (bbl)", row=1, col=2)
    fig_diag.update_yaxes(title_text="Oil rate (bbl/d)", row=1, col=2)

    fig_diag.update_layout(height=420, showlegend=False, **PLOT_TEMPLATE_LAYOUT)
    fig_diag.update_annotations(font=dict(color=TEXT_DARK))
    st.plotly_chart(fig_diag, use_container_width=True)

# ---------------------------------------------------------------- tab 3: anomalies
with tab3:
    st.markdown(f"### {len(well_anom)} anomalous readings detected for {well}")
    if len(well_anom):
        st.dataframe(
            well_anom[["date", "oil_rate_bbl", "arps_rate_bbl", "residual_pct", "likely_cause"]]
            .rename(columns={
                "date": "Date", "oil_rate_bbl": "Actual (bbl/d)", "arps_rate_bbl": "Expected/Arps (bbl/d)",
                "residual_pct": "Residual (%)", "likely_cause": "Likely Cause"
            }).round(1),
            use_container_width=True, height=350
        )
    else:
        st.success("No anomalies flagged for this well — production tracking its natural decline trend.")

    st.markdown("### Field-wide anomaly summary")
    field_summary = anomalies.groupby("well_id").size().reset_index(name="anomaly_count").sort_values("anomaly_count", ascending=False)
    fig_field = go.Figure(go.Bar(x=field_summary["well_id"], y=field_summary["anomaly_count"], marker_color=RUST))
    fig_field.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
                             xaxis_title="Well", yaxis_title="Anomalous days", **PLOT_TEMPLATE_LAYOUT)
    st.plotly_chart(fig_field, use_container_width=True)

# ---------------------------------------------------------------- tab 4: model insights
with tab4:
    st.markdown("### Arps vs XGBoost — field-wide holdout accuracy")
    fig_comp = go.Figure()
    fig_comp.add_trace(go.Bar(x=comparison["well_id"], y=comparison["holdout_rmse"], name="Arps RMSE", marker_color=OLIVE))
    fig_comp.add_trace(go.Bar(x=comparison["well_id"], y=comparison["ml_holdout_rmse"], name="XGBoost RMSE", marker_color=AMBER))
    fig_comp.update_layout(barmode="group", height=380,
                            yaxis_title="90-day holdout RMSE (bbl/d)",
                            legend=dict(font=dict(color=TEXT_DARK)), **PLOT_TEMPLATE_LAYOUT)
    st.plotly_chart(fig_comp, use_container_width=True)

    win_rate = (comparison["ml_holdout_rmse"] < comparison["holdout_rmse"]).mean() * 100
    st.info(f"XGBoost outperformed the classical Arps fit on **{win_rate:.0f}%** of wells (lower 90-day holdout RMSE).")

    st.markdown("### What drives the ML forecast? (feature importance)")
    fig_imp = go.Figure(go.Bar(x=importance["importance"], y=importance["feature"], orientation="h", marker_color=COPPER))
    fig_imp.update_layout(height=420, margin=dict(l=10,r=10,t=10,b=10),
                           yaxis=dict(autorange="reversed", gridcolor=BORDER_GRAY), xaxis_title="Importance",
                           **{k: v for k, v in PLOT_TEMPLATE_LAYOUT.items() if k != "yaxis"})
    st.plotly_chart(fig_imp, use_container_width=True)

st.markdown("---")
st.caption("Bronze → Silver → Gold pipeline in DuckDB · Classical Arps (SciPy) vs XGBoost · Isolation Forest anomaly detection · Built with Streamlit")

## Launch the app from Colab

Get a **free** authtoken from https://dashboard.ngrok.com/get-started/your-authtoken (sign up takes 30 seconds),
paste it into the cell below, then run it. It will print a public URL you can open directly.

In [ ]:
from pyngrok import ngrok, conf
import subprocess, time

# 1) Get a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
#    and paste it below (only needs to be run once per Colab session).
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# kill any previous tunnels/streamlit processes from earlier reruns in THIS session
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(6)

# Free ngrok accounts only allow ONE active tunnel at a time. If a previous Colab
# session disconnected without cleanly closing its tunnel, ngrok's servers may still
# think that old tunnel is "online" (ERR_NGROK_334) even though ngrok.kill() above
# only stops the LOCAL process. Retry a few times, since these stale tunnels usually
# expire on their own within a minute or two.
public_url = None
for attempt in range(5):
    try:
        public_url = ngrok.connect(8501)
        break
    except Exception as e:
        if "ERR_NGROK_334" in str(e) or "already online" in str(e):
            print(f"Attempt {attempt+1}/5: a previous tunnel is still live on ngrok's side, retrying in 15s...")
            print("If this keeps failing, go to https://dashboard.ngrok.com/endpoints and click 'Stop' "
                  "on the online endpoint, then rerun this cell.")
            time.sleep(15)
        else:
            raise

if public_url:
    print("🚀 Your Smart DCA Dashboard is live at:")
    print(public_url)
else:
    print("❌ Could not start the tunnel after 5 attempts.")
    print("Go to https://dashboard.ngrok.com/endpoints, stop the online endpoint, then rerun this cell.")
